# RF-DETR 1.7.0: Zero-Boilerplate Loading and Mobile Export

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/rf-detr/blob/develop/notebooks/release-demo_1-7.ipynb)

**What you'll learn:**
- Fine-tune RF-DETR Small on a small medical dataset (Axial MRI, 371 greyscale scans)
- Reload any checkpoint with `rfdetr.from_checkpoint()` — no class name or `num_classes` needed
- Export to TFLite (FP32, FP16, INT8) for mobile and embedded-edge deployment
- Run TFLite inference with only `numpy`, `tflite-runtime`, and `Pillow` — no PyTorch required

**Sections:**
1. [Install RF-DETR 1.7.0](#1-install-rf-detr-170)
2. [Config](#2-config)
3. [Dataset — Axial MRI](#3-dataset--axial-mri)
4. [Training](#4-training)
5. [`from_checkpoint()` — zero-boilerplate loading](#5-from_checkpoint--zero-boilerplate-loading)
6. [TFLite export — mobile and embedded deployment](#6-tflite-export--mobile-and-embedded-deployment)
7. [Next steps](#7-next-steps)

**RF-DETR** is a real-time object detection model that combines the accuracy of
transformer-based detectors with inference speeds suitable for production.
This notebook demonstrates two capabilities shipped in 1.7.0:

1. **`rfdetr.from_checkpoint()`** — load any user-trained checkpoint in one line,
   without knowing the model class or `num_classes` ahead of time
2. **`model.export(format="tflite")`** — deploy to Android, Raspberry Pi, and other
   edge targets via TFLite, with optional INT8 quantization for further size reduction

We train on the [Axial MRI](https://universe.roboflow.com/roboflow-100/axial-mri)
dataset — 371 greyscale MRI scans in two classes — to show the full pipeline on a
real medical imaging task.  RF-DETR 1.7.0 also ships `num_channels=1` for models
that ingest truly single-channel sensor streams; see the docs for that path.

> **Breaking change:** `peft` is no longer installed by default.
> If you use LoRA fine-tuning, install `pip install 'rfdetr[lora]'` separately.

## 1. Install RF-DETR 1.7.0

`[train,loggers]` brings the full training stack including PyTorch Lightning.
`[onnx]` is required for TFLite export — RF-DETR routes through ONNX via
`onnx2tf` before producing the `.tflite` files.

In [ ]:
!pip install -q 'rfdetr[train,loggers,onnx,tflite]==1.7.0.rc0' roboflow ai_edge_litert sng4onnx

## 2. Config

All notebook-level knobs in one place. Adjust `EPOCHS` and `BATCH_SIZE` to
match your hardware — every downstream cell reads from these variables.

`num_workers` is set to `os.cpu_count()` inside a Jupyter/Colab kernel where
process forking is safe, and to `0` when running as a plain Python script.
On macOS and Windows, spawn-based multiprocessing would otherwise re-import
this module as `__main__` and retrigger training.

In [ ]:
import os
from pathlib import Path

OUTPUT_DIR = "output"
BATCH_SIZE = 8   # reduce to 4 on Colab CPU
EPOCHS = 20
THRESHOLD = 0.3

os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    from IPython import get_ipython

    _in_notebook = get_ipython() is not None
except Exception:
    _in_notebook = False

# Outside a notebook kernel, macOS/Windows spawn-based multiprocessing will
# re-import this script as __main__, triggering training again.
# Use 0 workers in that case; inside a kernel the usual forking rules apply safely.
num_workers = os.cpu_count() if _in_notebook else 0

## 3. Dataset — Axial MRI

[Axial MRI](https://universe.roboflow.com/roboflow-100/axial-mri) from the
Roboflow 100 benchmark — 371 MRI scans across two classes: `negative` and
`positive`.  The images are greyscale medical scans stored as JPEG (three
channels), so no preprocessing is needed — we train directly on them.

Set `ROBOFLOW_API_KEY` as a Colab secret (Secrets panel, key icon) or as an
environment variable before running this cell. Any Roboflow API key works —
the dataset is public.

In [ ]:
import json

from roboflow import Roboflow

try:
    from google.colab import userdata

    API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    API_KEY = os.environ["ROBOFLOW_API_KEY"]

rf = Roboflow(api_key=API_KEY)
dataset = rf.workspace("roboflow-100").project("axial-mri").version(2).download("coco", location="datasets")
DATASET_DIR = dataset.location

with open(Path(DATASET_DIR) / "train" / "_annotations.coco.json") as f:
    _ann = json.load(f)

# Sort by category id to match the index order the model will use
CLASS_NAMES = [c["name"] for c in sorted(_ann["categories"], key=lambda c: c["id"])]
NUM_CLASSES = len(CLASS_NAMES)
print(f"Dataset : {DATASET_DIR}")
print(f"Classes : {NUM_CLASSES} — {CLASS_NAMES}")

with open(Path(DATASET_DIR) / "valid" / "_annotations.coco.json") as f:
    _val_ann = json.load(f)

val_images_dir = Path(DATASET_DIR) / "valid"
val_image_files = [img["file_name"] for img in _val_ann["images"]]

## 4. Training

In [ ]:
from rfdetr import RFDETRSmall

model = RFDETRSmall(  # type: ignore[no-untyped-call]
    num_classes=NUM_CLASSES,
    pretrain_weights="rf-detr-small.pth",
)
model.train(  # type: ignore[no-untyped-call]
    dataset_dir=DATASET_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=4,   # effective batch = BATCH_SIZE * 4
    lr=1e-4,
    num_workers=num_workers,
    output_dir=OUTPUT_DIR,
    use_ema=True,
    run_test=False,
    progress_bar="tqdm",
    seed=42,
)

### Predict on a test image

We load one validation image and run the trained model.  The MRI scans are
greyscale by nature, so we display them with `cmap="gray"` — but pass the
original PIL image (RGB mode) straight to `predict()`.
After training, the val images are also converted to true grayscale on disk
to demonstrate what the dataset looks like at the sensor level.

In [ ]:
%matplotlib inline

import numpy as np
import supervision as sv
from PIL import Image
import matplotlib.pyplot as plt

# Convert validation images to grayscale on disk for display purposes
for _p in val_images_dir.iterdir():
    if _p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        Image.open(_p).convert("L").convert("RGB").save(_p)

image = Image.open(val_images_dir / val_image_files[0])
print(f"Image mode: {image.mode}, size: {image.size}")

detections: sv.Detections = model.predict(image, threshold=THRESHOLD)

annotated = sv.BoxAnnotator().annotate(image.copy(), detections)
annotated = sv.LabelAnnotator().annotate(
    annotated, detections, labels=[CLASS_NAMES[c] for c in detections.class_id]
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(np.array(image), cmap="gray")
axes[0].axis("off")
axes[0].set_title("Input (greyscale MRI)")
axes[1].imshow(np.array(annotated))
axes[1].axis("off")
axes[1].set_title(f"Detected {len(detections)} region(s)")
plt.tight_layout()
plt.show()

## 5. `from_checkpoint()` — zero-boilerplate loading

Before 1.7.0, loading a user-trained checkpoint required knowing which model
class was used and passing `num_classes` explicitly — the checkpoint itself
carried no record of either.  `rfdetr.from_checkpoint()` reads both from the
checkpoint and instantiates the right class automatically.

In [ ]:
# Before 1.7.0 — you had to know the class and num_classes ahead of time:
# from rfdetr import RFDETRSmall
# model = RFDETRSmall(
#     pretrain_weights=f"{OUTPUT_DIR}/checkpoint_best_total.pth",
#     num_classes=NUM_CLASSES,
# )

In [ ]:
import rfdetr

# model_class and num_classes are inferred from the checkpoint automatically
loaded_model = rfdetr.from_checkpoint(f"{OUTPUT_DIR}/checkpoint_best_total.pth")
print(f"Loaded: {loaded_model.__class__.__name__}")
print(f"Classes: {loaded_model.class_names}")   # attribute populated from the saved checkpoint

### Confirm the loaded model produces the same detections

Running inference with the freshly loaded model on the same image verifies
that the checkpoint round-trips correctly — same weights, same results.

In [ ]:
detections2 = loaded_model.predict(image, threshold=THRESHOLD)
print(f"Original model: {len(detections)} detection(s)")
print(f"Loaded model:   {len(detections2)} detection(s)")

## 6. TFLite export — mobile and embedded deployment

`model.export(format="tflite")` converts the trained model to TFLite via an
ONNX intermediate graph.  Two files are always produced: FP32 (full precision)
and FP16 (half precision, ~2× smaller).  INT8 quantization is a separate call
and requires a directory of representative calibration images drawn from the
same distribution as training data.

> **Note:** the export cell below will be **silent for 5–15 minutes** while
> `onnx2tf` lowers the DINOv2 transformer graph to TensorFlow ops.
> This is expected — it has not hung.  A completion message appears when done.

In [ ]:
model.export(
    format="tflite",
    output_dir=OUTPUT_DIR,   # FP32 + FP16 .tflite files land here
)

In [ ]:
import glob

# List all generated .tflite files with their sizes on disk
tflite_files = glob.glob(f"{OUTPUT_DIR}/**/*.tflite", recursive=True)
for tflite_path in sorted(tflite_files):
    size_mb = Path(tflite_path).stat().st_size / 1e6
    print(f"  {tflite_path}  ({size_mb:.1f} MB)")

### TFLite inference helpers

Three reusable functions cover the full inference pipeline — no PyTorch or
RF-DETR required at runtime, just `tflite-runtime`, `numpy`, and `Pillow`:

- **`create_interpreter(model_path)`** — load and allocate a TFLite model
- **`run_inference(interp, image_path, threshold)`** — preprocess, invoke, decode
- **`visualize_detections(detections, image, class_names, title)`** — annotate and display

The same three calls work for FP32, FP16, and INT8 — swap the model path to
compare quantization variants side by side.

**Input contract** (NHWC `float32`, caller-normalized):
- Shape: `[1, H, W, C]` — onnx2tf converts NCHW → NHWC at export time
- Normalization: ImageNet statistics (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`)

**Outputs** (ONNX names preserved by onnx2tf):
- `dets [1, Q, 4]` — normalized cx, cy, w, h
- `labels [1, Q, num_classes+1]` — raw logits

In [ ]:
%matplotlib inline

import json
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import supervision as sv
from PIL import Image as PILImage

try:
    import tflite_runtime.interpreter as _tflite  # lightweight; preferred on edge devices

    _Interpreter = _tflite.Interpreter
except ImportError:
    import tensorflow as _tf                       # full TF is pre-installed on Colab

    _Interpreter = _tf.lite.Interpreter

In [ ]:
def _softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def create_interpreter(model_path: str | Path) -> Any:
    """Create and allocate a TFLite interpreter.

    Args:
        model_path: Path to the `.tflite` model file.

    Returns:
        An allocated TFLite interpreter ready for inference.

    Examples:
        >>> interp = create_interpreter("model_float32.tflite")
    """
    interp = _Interpreter(model_path=str(model_path))
    interp.allocate_tensors()
    inp_det = interp.get_input_details()
    out_det = interp.get_output_details()
    print(f"Input  : {inp_det[0]['shape']}  {inp_det[0]['dtype'].__name__}")
    for od in out_det:
        print(f"Output : {od['shape']}  name={od['name']}")
    return interp


def run_inference(
    interp: Any,
    image_path: str | Path,
    threshold: float = 0.3,
) -> tuple[sv.Detections, PILImage.Image]:
    """Preprocess one image, run TFLite inference, and decode detections.

    Args:
        interp: Allocated TFLite interpreter returned by `create_interpreter`.
        image_path: Path to the input image.
        threshold: Confidence threshold; detections below this are discarded.

    Returns:
        A tuple of (detections in pixel xyxy format, original PIL image).

    Examples:
        >>> interp = create_interpreter("model.tflite")
        >>> dets, img = run_inference(interp, "image.jpg", threshold=0.3)
    """
    inp_det = interp.get_input_details()
    out_det = interp.get_output_details()
    _, H, W, C = inp_det[0]["shape"]

    _imagenet_mean = [0.485, 0.456, 0.406]
    _imagenet_std  = [0.229, 0.224, 0.225]
    mean = np.array([_imagenet_mean[i % 3] for i in range(C)], dtype=np.float32)
    std  = np.array([_imagenet_std[i % 3]  for i in range(C)], dtype=np.float32)

    pil_img = PILImage.open(image_path)
    pil_mode = "L" if C == 1 else "RGB"
    arr = np.array(pil_img.convert(pil_mode).resize((W, H)), dtype=np.float32) / 255.0
    if arr.ndim == 2:                              # "L" → (H, W); TFLite needs (H, W, 1)
        arr = arr[:, :, np.newaxis]
    inp_tensor = (arr - mean) / std

    interp.set_tensor(inp_det[0]["index"], inp_tensor[np.newaxis])
    interp.invoke()

    # RF-DETR ONNX output names: "dets" = pred_boxes, "labels" = pred_logits.
    # Match by name so the code is robust to onnx2tf output reordering.
    boxes_idx  = next(i for i, od in enumerate(out_det) if "dets"   in od["name"])
    logits_idx = next(i for i, od in enumerate(out_det) if "labels" in od["name"])
    boxes_cwh = interp.get_tensor(out_det[boxes_idx]["index"])[0]   # (Q, 4) normalized cxcywh
    logits    = interp.get_tensor(out_det[logits_idx]["index"])[0]  # (Q, num_classes+1)

    probs  = _softmax(logits[:, :-1])              # drop background (last logit)
    scores = probs.max(axis=-1)
    cls    = probs.argmax(axis=-1)
    keep   = scores > threshold

    cx, cy, bw, bh = boxes_cwh[keep].T
    ow, oh = pil_img.size
    xyxy = np.stack([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], axis=1)
    xyxy *= np.array([ow, oh, ow, oh], dtype=np.float32)

    return sv.Detections(xyxy=xyxy, confidence=scores[keep], class_id=cls[keep].astype(int)), pil_img


def visualize_detections(
    detections: sv.Detections,
    image: PILImage.Image,
    class_names: list[str],
    title: str = "",
) -> None:
    """Annotate an image with detections and display it.

    Args:
        detections: Detections in pixel xyxy format.
        image: PIL image corresponding to the detections.
        class_names: Class name list indexed by `class_id`.
        title: Plot title shown above the image.

    Examples:
        >>> interp = create_interpreter("model.tflite")
        >>> dets, img = run_inference(interp, "image.jpg")
        >>> visualize_detections(dets, img, ["negative", "positive"], title="FP32")
    """
    annotated = sv.BoxAnnotator().annotate(image.convert("RGB").copy(), detections)
    annotated = sv.LabelAnnotator().annotate(
        annotated, detections, labels=[class_names[c] for c in detections.class_id]
    )
    plt.figure(figsize=(8, 6))
    plt.imshow(np.array(annotated))
    plt.axis("off")
    plt.title(title)
    plt.show()
    print(f"{title}: {len(detections)} detection(s)")

In [ ]:
# class names are stored in training_config.json by RF-DETR 1.7.0
_class_names: list[str] = json.loads(
    (Path(OUTPUT_DIR) / "training_config.json").read_text()
)["class_names"]
_image_path = val_images_dir / val_image_files[0]
print(f"Classes ({len(_class_names)}): {_class_names}")
print(f"Image   : {_image_path}")

### FP32 inference

In [ ]:
_fp32_path = next(Path(OUTPUT_DIR).glob("**/*_float32.tflite"))
_interp_fp32 = create_interpreter(_fp32_path)
_dets_fp32, _img = run_inference(_interp_fp32, _image_path, THRESHOLD)
visualize_detections(_dets_fp32, _img, _class_names, title=f"TFLite FP32 (threshold={THRESHOLD})")

### FP16 inference

In [ ]:
_fp16_path = next(Path(OUTPUT_DIR).glob("**/*_float16.tflite"))
_interp_fp16 = create_interpreter(_fp16_path)
_dets_fp16, _img = run_inference(_interp_fp16, _image_path, THRESHOLD)
visualize_detections(_dets_fp16, _img, _class_names, title=f"TFLite FP16 (threshold={THRESHOLD})")

### INT8 export with calibration

INT8 quantization further halves the model size and accelerates inference on
edge hardware with dedicated INT8 units (Android NNAPI, Coral Edge TPU).
The `calibration_data` directory should contain images representative of your
deployment distribution — here we reuse the validation split.

In [ ]:
model.export(
    format="tflite",
    quantization="int8",
    calibration_data=str(val_images_dir),   # representative images for quantization calibration
    output_dir=OUTPUT_DIR,
)

### INT8 inference

In [ ]:
_int8_path = next(Path(OUTPUT_DIR).glob("**/*_int8.tflite"))
_interp_int8 = create_interpreter(_int8_path)
_dets_int8, _img = run_inference(_interp_int8, _image_path, THRESHOLD)
visualize_detections(_dets_int8, _img, _class_names, title=f"TFLite INT8 (threshold={THRESHOLD})")

## 7. Next steps

You now have a trained MRI detector, loaded from checkpoint in
one line, and exported to TFLite for edge deployment.  From here:

- [Multi-channel and custom sensor docs](https://rfdetr.roboflow.com) — extending to 4-channel RGBN or 16-channel hyperspectral
- [TFLite and export docs](https://rfdetr.roboflow.com) — Android NNAPI, Raspberry Pi, ONNX, TensorRT deployment
- [RF-DETR 1.5.0 notebook](https://colab.research.google.com/github/roboflow/rf-detr/blob/develop/notebooks/release-demo_1-5.ipynb) — custom augmentations
- [RF-DETR 1.6.0 notebook](https://colab.research.google.com/github/roboflow/rf-detr/blob/develop/notebooks/release-demo_1-6.ipynb) — PyTorch Lightning building blocks
- [Full changelog](https://github.com/roboflow/rf-detr/compare/1.6.5...1.7.0) — every commit between 1.6.5 and 1.7.0